# Week 23: Next-Gen AI Reasoning, MCTS & Test-Time Compute
## Notebook 05: Self-Correction & Backtracking Reasoning from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Standard LLMs suffer from cognitive stubbornness: once an error is produced in the context window, self-attention attends to that error, making the model double down on its mistake.
**Self-Correction & Backtracking:**
1. Maintain an explicit **Reasoning Stack**.
2. Periodically execute **Self-Verification Assertions**.
3. If an assertion fails: **Prune the broken reasoning branch**, revert the state, and explore an alternate hypothesis!

---

### 2. Architecture & State Rollback
$$\text{State } 0 \to \text{Hypothesis A} \xrightarrow{\text{Assertion: FAILED!}} \text{Backtrack to State } 0 \to \text{Hypothesis B} \xrightarrow{\text{PASSED!}} \text{Success}$$

In [ ]:
class BacktrackingReasoningEngine:
    def __init__(self, target_sum=15):
        self.target = target_sum
        self.history_stack = []

    def verify_partial_state(self, current_sum, remaining_steps):
        # Constraint check: if current sum already exceeds target, fail immediately!
        if current_sum > self.target:
            return False, "Exceeded target limit"
        return True, "Valid"

    def solve(self, candidates):
        # State: (current_sum, path_items)
        stack = [(0, [])]
        
        explored_count = 0
        while stack:
            explored_count += 1
            curr_sum, path = stack.pop()
            
            # Goal Check
            if curr_sum == self.target:
                return path, explored_count
                
            for num in candidates:
                new_sum = curr_sum + num
                is_valid, msg = self.verify_partial_state(new_sum, len(path))
                
                if is_valid:
                    # Valid partial branch: push to stack
                    stack.append((new_sum, path + [num]))
                else:
                    # Backtrack trigger: prune branch immediately!
                    pass
                    
        return None, explored_count

engine = BacktrackingReasoningEngine(target_sum=15)
solution, steps = engine.solve([7, 5, 3])

print("--- Backtracking Reasoning Execution Trace ---")
print(f"Goal: Find combinations of [7, 5, 3] that sum to 15")
print(f"Discovered Valid Path: {solution}")
print(f"Total Explored States: {steps} (Branch pruning prevented search explosion!)")

### 3. Step-by-Step Code Walkthrough
1. **Verification Gate:** Checks partial state validity before deep exploration.
2. **Immediate Pruning:** Invalid candidate branches ($>15$) are discarded from the stack.
3. **Backtracking:** Reverts state cleanly to explore alternate branches until reaching the goal.

---

### 4. Industry Applications & Trade-offs
- **Automated Theorem Proving (Lean 4):** Backtracking search over mathematical proof tactics.
- **Compiler Optimization:** Exploring instruction reordering graphs.

---

### 5. Top Interview Questions & Answers
**Q1: Why do standard LLMs struggle to self-correct without external verification tools?**
> **Answer:** Huang et al. (2023) demonstrated that LLMs without external feedback or verifiers rarely correct their own errors; prompting a model to "review your work" often causes it to change correct answers into incorrect ones (sycophancy). True self-correction requires deterministic verification (interpreters, unit tests, or PRMs).

**Q2: What is the role of explicit state backtracking in reducing token consumption?**
> **Answer:** In linear reasoning, an LLM generates thousands of tokens wandering down a broken path. Backtracking truncates the context window back to the last known valid state checkpoint, saving context tokens and preventing hallucination pollution.